# Rubric Dimensions - Schema Quality, Consistency, Validity, Freshness + Composite Scoring

This notebook implements and validates the four rubric dimensions not covered by
the existing header-detection / core-profiling / outlier / PII checks
(**Schema Quality**, **Consistency**, **Validity**, **Freshness**), plus
the **composite Data Quality Score** that combines all 8 rubric dimensions into a
single weighted number, alongside a separate **Privacy Risk** report.

These map to the teacher's separate 8-dimension rubric (weights below), which is
not the same numbering as `plan.md`'s own "Suggested Build Order" (Section 10) --
`plan.md`'s own items 5 and 6 are `format_consistency.py` and `encoding.py`, which
are different, not-yet-built modules. This notebook is intentionally not numbered
against that list to avoid confusion.

Weights: Completeness 20%, Validity 20%, Type Reliability 15%, Consistency 15%,
Uniqueness 10%, Schema Quality 10%, Outlier Risk 5%, Freshness 5%.


## Introduction

A dataset can pass every Task 1-4 check (no missing values, no duplicates,
correct types, no PII leaks) and still be low quality in ways those checks
don't catch:

- **Schema Quality** - column *names* themselves can be broken (`Unnamed: 7`,
  duplicate names after a copy-paste, vague placeholders like `Column1`) even
  when every value in the column is perfectly fine.
- **Consistency** - the same real-world value written more than one way
  (`Paid` / `paid` / `PAID`) silently breaks any `group by` or `value_counts()`.
- **Validity** - a value can be present, correctly typed, and still wrong: a
  negative quantity, a date that doesn't parse, an email with no `@`, a
  delivery date before the order date.
- **Freshness** - the data can be internally perfect and still be *stale* --
  useless for a "current state" report if the newest row is 8 months old.

And finally: a dataset can be analytically high quality but still unsafe to
share (PII present). The rubric is explicit that **Privacy Risk must never be
folded into the quality score** -- it's reported side by side, not subtracted.


## Methodology

**Schema Quality** (`engine/checks/schema_quality.py`) -- pure name inspection,
no value parsing: flags pandas/ingestion auto-generated names
(`Unnamed: N` / `unnamed_N`), duplicate column names (case-insensitive),
vague generic names (`Column1`, `Data`, `X`), and empty/whitespace names.
Deliberately does **not** flag legitimate short business codes (`PO`, `SKU`,
`VAT`).

**Consistency** (`engine/checks/consistency.py`) -- only runs on columns
`column_classifier.classify_columns()` marks `categorical` or `identifier`
(measurement/date/pii/free_text are skipped, with the skip reason recorded,
not silently). Normalizes each value (`strip().lower()` + whitespace
collapse), groups raw values by normalized form, and flags every row whose
raw value is a minority variant of a group's most common ("canonical") form.
Note: this catches case/whitespace variants only -- true abbreviation
collapsing (`Lahore` vs `LHR`) needs fuzzy matching and is out of Phase 1
scope.

**Validity** (`engine/checks/validity.py`) -- four independent sub-checks:
1. `measurement` columns whose *name* implies non-negative semantics
   (qty, quantity, count, age, stock, units, weight, hours, days) -> flag
   negative values.
2. `measurement` columns on Easby's known suspicious-zero list
   (`domain_rules.suspicious_zero_columns_present`) -> flag zero values.
3. `date` columns -> flag individual unparseable cells and implausible years
   (a column can be *mostly* valid dates and still have a handful of bad
   cells -- that's different from `type_mismatch.py`'s dominant-type check).
4. columns whose name suggests an email field -> flag values that don't match
   the same email pattern `detect_pii.py` already uses (imported, not
   duplicated).

Plus, at the frame level: cross-column date-order rules from
`domain_rules.default_cross_column_rules()` (e.g. Expected Delivery Date must
be >= Order Date).

**Freshness** (`engine/checks/freshness.py`) -- column-level, not row-level:
for each `date` column, compare the most recent value against a configurable
threshold (`SETTINGS["freshness_days"]`, default 90 days).

**Scoring** (`engine/scoring.py`) -- `compute_data_quality_score()`. Important
design decision: existing checks' `CheckResult.dimension` field
(`"validity"` on both `type_mismatch.py` and `outliers.py`) does **not**
match the rubric's 8 separate dimensions, so the scorer takes results
**explicitly keyed by rubric dimension name** rather than trusting that
field. A dimension with no results supplied is excluded from the composite
*transparently* (`scorable_weight_fraction` reports what % of total rubric
weight the score actually covers), never silently treated as 0/failing.


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import pandas as pd

# Allow notebook execution from notebooks/ without installing the package
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_quality_engine.engine.column_classifier import classify_columns
from data_quality_engine.engine.checks.missing_values import check_missing_values
from data_quality_engine.engine.checks.duplicates import check_duplicates
from data_quality_engine.engine.checks.type_mismatch import check_type_consistency_frame
from data_quality_engine.engine.checks.outliers import detect_outliers_frame
from data_quality_engine.engine.checks.schema_quality import check_schema_quality
from data_quality_engine.engine.checks.consistency import check_consistency_frame
from data_quality_engine.engine.checks.validity import check_validity_frame
from data_quality_engine.engine.checks.freshness import check_freshness_frame
from data_quality_engine.engine.pii.detect_pii import detect_pii_in_series
from data_quality_engine.engine.scoring import compute_data_quality_score


## Demonstration Data

Synthetic, small on purpose (so every issue below is visible by eye in the
printed tables), but the column names and issue types mirror what actually
shows up in the real Easby files:

- `Unnamed: 7` -- the pandas placeholder name seen on real sheets like
  `Sheet4` in the Invoice List workbook.
- `Status` has a `Paid` / `paid` case variant.
- `Order Qty` has a negative value (invalid for a quantity column).
- `Standard Cost` has a suspicious `0.0` (a real cost shouldn't be free).
- `Order Date` has one unparseable cell (`"not-a-date"`).
- `Expected Del Date` vs `Order Date` has one cross-column violation (a
  delivery date earlier than its order date).
- `Contact Email` has one malformed value (`"bad-email"`).
- `Contact Email` also demonstrates the separate Privacy Risk report.


In [2]:
df_raw = pd.DataFrame(
    {
        "Invoice No": ["INV001", "INV002", "INV003", "INV004", "INV005", None],
        "Status": ["Paid", "paid", "Paid", "Paid", "Overdue", "Overdue"],
        "Order Qty": [10, -5, 20, 15, 8, 12],
        "Standard Cost": [1.25, 0.0, 3.40, 2.10, 1.90, 2.50],
        "Order Date": ["2024-01-05", "2024-01-10", "not-a-date", "2024-01-20", "2024-01-25", "2024-01-28"],
        "Expected Del Date": ["2024-01-01", "2024-01-20", "2024-01-15", "2024-01-25", "2024-02-01", "2024-02-05"],
        "Contact Email": ["a@easby.com", "bad-email", "c@easby.com", "d@easby.com", "e@easby.com", "f@easby.com"],
        "Unnamed: 7": [1, 2, 3, 4, 5, 6],
    }
)
df_raw


,Invoice No,Status,Order Qty,Standard Cost,Order Date,Expected Del Date,Contact Email,Unnamed: 7
0,INV001,Paid,10,1.25,2024-01-05,2024-01-01,a@easby.com,1
1,INV002,paid,-5,0.00,2024-01-10,2024-01-20,bad-email,2
2,INV003,Paid,20,3.40,not-a-date,2024-01-15,c@easby.com,3
3,INV004,Paid,15,2.10,2024-01-20,2024-01-25,d@easby.com,4
4,INV005,Overdue,8,1.90,2024-01-25,2024-02-01,e@easby.com,5
5,NaN,Overdue,12,2.50,2024-01-28,2024-02-05,f@easby.com,6


In [3]:
roles = classify_columns(df_raw)
roles


{'Invoice No': 'identifier',
 'Status': 'categorical',
 'Order Qty': 'measurement',
 'Standard Cost': 'measurement',
 'Order Date': 'date',
 'Expected Del Date': 'date',
 'Contact Email': 'pii',
 'Unnamed: 7': 'measurement'}

## Schema Quality

In [4]:
schema_results = check_schema_quality(df_raw)
for r in schema_results:
    print(f"{r.column:20} status={r.status:8} issues={r.issues_found}  {r.details.get('issues')}")


Invoice No           status=passed   issues=0  []
Status               status=passed   issues=0  []
Order Qty            status=passed   issues=0  []
Standard Cost        status=passed   issues=0  []
Order Date           status=passed   issues=0  []
Expected Del Date    status=passed   issues=0  []
Contact Email        status=passed   issues=0  []
Unnamed: 7           status=failed   issues=1  ['auto_generated_name']


## Consistency

In [5]:
consistency_results = check_consistency_frame(df_raw, roles=roles)
for r in consistency_results:
    detail = r.details.get("examples") or r.details.get("reason")
    print(f"{r.column:20} status={r.status:8} issues={r.issues_found}  {detail}")


Invoice No           status=passed   issues=0  None
Status               status=failed   issues=1  [{'normalized': 'paid', 'canonical': 'Paid', 'variants': {'Paid': 3, 'paid': 1}}]
Order Qty            status=passed   issues=0  skipped_non_categorical_column
Standard Cost        status=passed   issues=0  skipped_non_categorical_column
Order Date           status=passed   issues=0  skipped_non_categorical_column
Expected Del Date    status=passed   issues=0  skipped_non_categorical_column
Contact Email        status=passed   issues=0  skipped_non_categorical_column
Unnamed: 7           status=passed   issues=0  skipped_non_categorical_column


## Validity

In [6]:
validity_rules = [
    {"name": "expected_del_after_order", "left": "Expected Del Date", "op": ">=", "right": "Order Date"}
]
validity_results = check_validity_frame(df_raw, roles=roles, cross_column_rules=validity_rules)
for r in validity_results:
    rule = r.details.get("rule", r.details.get("reason"))
    print(f"{str(r.column):32} status={r.status:8} issues={r.issues_found}  {rule}")


Invoice No                       status=passed   issues=0  no_validity_rule_for_role
Status                           status=passed   issues=0  no_validity_rule_for_role
Order Qty                        status=failed   issues=1  negative_value_not_allowed
Standard Cost                    status=failed   issues=1  suspicious_zero_value
Order Date                       status=failed   issues=1  invalid_or_implausible_date
Expected Del Date                status=passed   issues=0  invalid_or_implausible_date
Contact Email                    status=failed   issues=1  invalid_email_format
Unnamed: 7                       status=passed   issues=0  no_validity_rule_for_role
Expected Del Date vs Order Date  status=failed   issues=1  cross_column_date_rule:expected_del_after_order


## Freshness

In [7]:
# as_of is fixed for a reproducible demo -- in main.py this defaults to datetime.now()
freshness_results = check_freshness_frame(df_raw, roles=roles, as_of=datetime(2024, 6, 1))
for r in freshness_results:
    detail = r.details.get("reason") or f"max_date={r.details.get('max_date')} days_since_max={r.details.get('days_since_max')}"
    print(f"{r.column:20} status={r.status:8} issues={r.issues_found}  {detail}")


Invoice No           status=passed   issues=0  skipped_non_date_column
Status               status=passed   issues=0  skipped_non_date_column
Order Qty            status=passed   issues=0  skipped_non_date_column
Standard Cost        status=passed   issues=0  skipped_non_date_column
Order Date           status=failed   issues=1  max_date=2024-01-28 days_since_max=125
Expected Del Date    status=failed   issues=1  max_date=2024-02-05 days_since_max=117
Contact Email        status=passed   issues=0  skipped_non_date_column
Unnamed: 7           status=passed   issues=0  skipped_non_date_column


## Composite Scoring

Reusing the already-implemented Task 2/3 checks for the other 4 rubric
dimensions (Completeness, Type Reliability, Uniqueness, Outlier Risk) so this
demo produces a full 8-dimension score, plus the Task 4 PII summary for the
separate Privacy Risk report.


In [8]:
missing = check_missing_values(df_raw)
dup = check_duplicates(df_raw)
type_results = check_type_consistency_frame(df_raw)
outlier_results = detect_outliers_frame(df_raw)
pii_summary = {str(c): detect_pii_in_series(df_raw[c]) for c in df_raw.columns}

dimension_results = {
    "completeness": missing,
    "type_reliability": type_results,
    "uniqueness": dup,
    "outlier_risk": outlier_results,
    "schema_quality": schema_results,
    "consistency": consistency_results,
    "validity": validity_results,
    "freshness": freshness_results,
}

score = compute_data_quality_score(dimension_results, pii_summary_by_column=pii_summary)

print("Data Quality Score:", score["data_quality_score"], "/ 100")
print("Scorable weight fraction:", score["scorable_weight_fraction"])
print()
for dim, info in score["dimension_scores"].items():
    print(f"{dim:18} score={str(info['score']):8} weight={info['weight']:.2f} available={info['available']}")
print()
print("Privacy Risk (separate, never part of the score above):")
print(score["privacy_risk"])


Data Quality Score: 80.35 / 100
Scorable weight fraction: 1.0

completeness       score=87.5     weight=0.20 available=True
validity           score=44.44    weight=0.20 available=True
type_reliability   score=100.0    weight=0.15 available=True
consistency        score=87.5     weight=0.15 available=True
uniqueness         score=100.0    weight=0.10 available=True
schema_quality     score=87.5     weight=0.10 available=True
outlier_risk       score=66.67    weight=0.05 available=True
freshness          score=75.0     weight=0.05 available=True

Privacy Risk (separate, never part of the score above):
{'risk_level': 'medium', 'columns_with_pii': 1, 'total_columns': 8, 'columns_flagged': ['Contact Email'], 'pii_types_found': ['EMAIL'], 'note': 'Reported separately -- never subtracted from data_quality_score.'}


## Validation Checks

Confirming each planted issue was actually caught, and that the composite
score / privacy risk fields are populated as expected.


In [9]:
def first(results, col):
    return [r for r in results if r.column == col][0]

by_col_schema = {r.column: r for r in schema_results}
assert by_col_schema["Unnamed: 7"].status == "failed", "Unnamed column should be flagged"

by_col_consistency = {r.column: r for r in consistency_results}
assert by_col_consistency["Status"].status == "failed", "Paid/paid variant should be flagged"

assert first(validity_results, "Order Qty").status == "failed", "negative qty should be flagged"
assert first(validity_results, "Standard Cost").status == "failed", "suspicious zero cost should be flagged"
assert first(validity_results, "Order Date").status == "failed", "unparseable date should be flagged"
assert first(validity_results, "Contact Email").status == "failed", "bad email should be flagged"
assert first(validity_results, "Expected Del Date vs Order Date").status == "failed", "date-order violation should be flagged"

assert first(freshness_results, "Order Date").status == "failed", "stale date column should be flagged"

assert score["data_quality_score"] is not None
assert score["scorable_weight_fraction"] == 1.0, "all 8 dimensions were supplied in this demo"
assert score["privacy_risk"]["columns_with_pii"] == 1
assert "EMAIL" in score["privacy_risk"]["pii_types_found"]

print("All assertions passed.")


All assertions passed.


## Results Summary

| Check | Planted issue | Caught? |
|---|---|---|
| Schema Quality | `Unnamed: 7` placeholder column name | Yes |
| Consistency | `Paid` / `paid` case variant in `Status` | Yes |
| Validity - non-negative | `-5` in `Order Qty` | Yes |
| Validity - suspicious zero | `0.0` in `Standard Cost` | Yes |
| Validity - date parse | `"not-a-date"` in `Order Date` | Yes |
| Validity - email format | `"bad-email"` in `Contact Email` | Yes |
| Validity - cross-column | Expected Del Date earlier than Order Date | Yes |
| Freshness | `Order Date` column stale as of the fixed `as_of` date | Yes |
| Privacy Risk | 1 PII column (`Contact Email`), reported separately | Yes |

All 8 rubric dimensions were scorable in this demo
(`scorable_weight_fraction == 1.0`), which only happens when every
dimension's checks were actually run and passed at least one non-error
result -- in `main.py`'s real pipeline this depends on the sheet actually
having, e.g., a recognizable date column.


## Limitations

- **Consistency** only normalizes case/whitespace -- true abbreviation
  collapsing (`Lahore` vs `LHR`) needs fuzzy matching (`rapidfuzz`), which is
  a stretch goal, not implemented here.
- **Validity's** non-negative check is name-based (`qty`, `age`, ...) -- a
  genuinely non-negative column with an unrecognized name won't be checked,
  and a column whose name happens to match but is legitimately signed
  (rare) could be a false positive. Same trade-off `Company Reg No.` /
  `Fax` exposed earlier in PII detection: name-hint heuristics are fast and
  explainable but not perfect.
- **Freshness** is column-level (one verdict per date column), not a
  finer-grained "are there gaps in the middle of the range" check (e.g. a
  missing month) -- that's listed as a stretch example in the rubric, not
  core Phase 1 scope.
- **Column classification** (`classify_columns`) has to guess a column's
  role from name + values; on a real sheet with unusual column names, a
  `date` column can occasionally be misclassified as `identifier` (seen
  during testing against `sample_data.xlsx`) -- when that happens, the
  date-specific Validity/Freshness rules simply don't fire for that column,
  rather than firing incorrectly. Worth spot-checking classification output
  on new files rather than trusting it blindly.


## Conclusion

Schema Quality, Consistency, Validity, and Freshness close the gap between
Task 1-4 and the teacher's full 8-dimension rubric. Combined with the
already-implemented Completeness, Type Reliability, Uniqueness, and Outlier
Risk checks, `compute_data_quality_score()` now produces a single composite
Data Quality Score with full transparency about which dimensions actually
contributed to it -- and Privacy Risk is reported alongside it, never
subtracted from it, exactly as the rubric specifies.

Not yet done: wiring a configurable rule set for cross-column date rules per
file (currently the demo passes rules in manually; `main.py` uses
`domain_rules.default_cross_column_rules()` as the default), and a PDF/XLSX
export of this report (still on the `requirements-future.txt` list).
